# quant

Tiện ích lượng tử hóa trọng số theo từng lớp (dùng cho Thí nghiệm B, C).

Hai chế độ co giãn (scaling) cho các định dạng có dải cố định (posit, FP8):
  - "none" : làm tròn trực tiếp giá trị gốc (rounding quantization, như [07]).
  - "pow2" : nhân trọng số của mỗi lớp với 2^s trước khi làm tròn rồi chia lại,
             s là số nguyên chọn để sai số bình phương nhỏ nhất (giống layer-wise
             scaling của [08] và "exponent bias" của [40]; vì là lũy thừa 2 nên
             trong phần cứng chỉ là cộng/trừ vào số mũ).
INT n-bit luôn cần hệ số co giãn: "max" (s = max|W|/(2^(n-1)-1)) hoặc
"mse" (dò hệ số cắt α·max để sai số bình phương nhỏ nhất — hiệu chỉnh chuẩn).

## Mục tiêu và cách đọc

Đây là thư viện hỗ trợ, không phải thí nghiệm độc lập. Các notebook 01–11 tự nhập thư viện này. Bật RUN_EXPERIMENT để nạp các định nghĩa khi muốn tìm hiểu.

## Cách sử dụng

1. Chọn kernel Python có `ipykernel`, `numpy`, `torch`, `matplotlib`.
2. Giữ `RUN_EXPERIMENT = False` và chọn **Run All** để xem dữ liệu sẵn có.
3. Đổi thành `True` nếu muốn thực hiện lại; chạy tuần tự các ô từ đầu.
4. **Ctrl+S / Cmd+S** lưu toàn bộ output vào notebook. Báo cáo HTML và CSV được xuất tự động ở ô cuối.

**Lưu ý:** các hình B/C/D là file đã lưu; sau khi đổi kết quả, chạy notebook 08 để vẽ lại trước khi dùng hình trong bài. Thời điểm xuất báo cáo không phải thời điểm thực hiện thí nghiệm gốc.


## 1. Thiết lập và chọn chế độ

Cài môi trường một lần theo `HUONG_DAN_NOTEBOOK.md`. Mỗi lần chạy tạo thư mục riêng trong `results/notebook_runs/`. Khi chạy lại thí nghiệm, các kết quả chuẩn và checkpoint hiện có được sao lưu trước.

In [ ]:
from pathlib import Path
import os
import sys

# Tìm 02_demo dù kernel bắt đầu ở thư mục workspace hoặc thư mục notebook.
candidates = [Path.cwd(), *Path.cwd().parents]
DEMO_ROOT = next((base / suffix for base in candidates
                  for suffix in ("", "02_demo", "Gk_XLDL/02_demo")
                  if (base / suffix / "posit_lab/numformats.py").is_file()), None)
if DEMO_ROOT is None:
    raise FileNotFoundError("Mở notebook trong workspace chứa Gk_XLDL/02_demo.")
DEMO_ROOT = DEMO_ROOT.resolve()
os.chdir(DEMO_ROOT)
if str(DEMO_ROOT) not in sys.path:
    sys.path.insert(0, str(DEMO_ROOT))
from notebook_tools.reporting import NotebookRun

# False: xem số liệu sẵn có. True: chạy code thí nghiệm bên dưới.
RUN_EXPERIMENT = False
SCRIPT_ARGS = []
__file__ = str(DEMO_ROOT / 'posit_lab/quant.py')
__package__ = 'posit_lab'
session = NotebookRun(DEMO_ROOT, 'posit_lab/quant.py', RUN_EXPERIMENT, SCRIPT_ARGS)
print("Thư mục lưu lần này:", session.out)

# Cell magic giữ code thẳng hàng; vẫn áp dụng chế độ chạy và lưu nhật ký.
from IPython.core.magic import register_cell_magic
import __future__

@register_cell_magic
def experiment(line, cell):
    if not RUN_EXPERIMENT:
        print("Đã bỏ qua tính toán; đặt RUN_EXPERIMENT = True để chạy ô này.")
        return
    with session.execute():
        code = compile(cell, str(session.source), "exec",
                       flags=__future__.annotations.compiler_flag)
        exec(code, get_ipython().user_ns)


## 2. pow2_shift, int_mse_scale, quantize_tensor

Mã chuyển nguyên từ `posit_lab/quant.py`. Ô chỉ thực thi khi `RUN_EXPERIMENT = True`; giữ thứ tự để các biến và hàm được khởi tạo đúng.

In [ ]:
%%experiment
"""Tiện ích lượng tử hóa trọng số theo từng lớp (dùng cho Thí nghiệm B, C).

Hai chế độ co giãn (scaling) cho các định dạng có dải cố định (posit, FP8):
  - "none" : làm tròn trực tiếp giá trị gốc (rounding quantization, như [07]).
  - "pow2" : nhân trọng số của mỗi lớp với 2^s trước khi làm tròn rồi chia lại,
             s là số nguyên chọn để sai số bình phương nhỏ nhất (giống layer-wise
             scaling của [08] và "exponent bias" của [40]; vì là lũy thừa 2 nên
             trong phần cứng chỉ là cộng/trừ vào số mũ).
INT n-bit luôn cần hệ số co giãn: "max" (s = max|W|/(2^(n-1)-1)) hoặc
"mse" (dò hệ số cắt α·max để sai số bình phương nhỏ nhất — hiệu chỉnh chuẩn).
"""
import copy

import torch

from .numformats import make_quantizer, quantize_int
from .model import LAYERS


def pow2_shift(w: torch.Tensor, qfn, lo: int = -12, hi: int = 12) -> int:
    best, best_s = None, 0
    for s in range(lo, hi + 1):
        sc = 2.0 ** s
        err = torch.sum((w - qfn(w * sc) / sc) ** 2).item()
        if best is None or err < best:
            best, best_s = err, s
    return best_s


def int_mse_scale(w: torch.Tensor, n: int) -> float:
    qmax = 2 ** (n - 1) - 1
    amax = w.abs().max().item()
    best, best_sc = None, amax / qmax
    for a in [1.0 - 0.02 * i for i in range(0, 40)]:  # α từ 1.0 xuống 0.22
        sc = a * amax / qmax
        err = torch.sum((w - quantize_int(w, n, sc)) ** 2).item()
        if best is None or err < best:
            best, best_sc = err, sc
    return best_sc


def quantize_tensor(w: torch.Tensor, fmt: str, scale: str = "none"):
    """Trả về (w_q, info) — info ghi lại hệ số co giãn đã dùng."""
    if fmt.startswith("int"):
        n = int(fmt[3:])
        sc = int_mse_scale(w, n) if scale == "mse" else None
        wq = quantize_int(w, n, sc)
        return wq, {"int_scale": sc if sc is not None else w.abs().max().item() / (2 ** (n - 1) - 1)}
    qfn = make_quantizer(fmt)
    if scale == "pow2":
        s = pow2_shift(w, qfn)
        return qfn(w * 2.0 ** s) / 2.0 ** s, {"pow2_shift": s}
    return qfn(w), {}




## 3. quantize_model_weights

Mã chuyển nguyên từ `posit_lab/quant.py`. Ô chỉ thực thi khi `RUN_EXPERIMENT = True`; giữ thứ tự để các biến và hàm được khởi tạo đúng.

In [ ]:
%%experiment
@torch.no_grad()
def quantize_model_weights(model, fmt: str, scale: str = "none"):
    """Bản sao mô hình với trọng số conv/fc đã lượng tử hóa (bias giữ FP32)."""
    m = copy.deepcopy(model)
    infos, rel_err = {}, {}
    for name in LAYERS:
        layer = getattr(m, name)
        w = layer.weight.data
        wq, info = quantize_tensor(w, fmt, scale)
        rel_err[name] = (torch.norm(w - wq) / torch.norm(w)).item()
        layer.weight.data.copy_(wq)
        infos[name] = info
    return m, infos, rel_err


## Kết quả, bảng số liệu và hình

Ô cuối đọc đầy đủ các trường JSON, xuất CSV UTF-8 cho Excel, lưu JSON gốc, nhúng hình vào HTML để mở độc lập. Các bảng dài có thanh cuộn. Khi viết bài, ghi rõ cấu hình, seed, tập dữ liệu và giới hạn mô phỏng.

In [ ]:
session.report([], [])